In [ ]:
from pathlib import Path
import numpy as np
import ezc3d
from scipy.interpolate import interp1d

ROOT = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk")

ANGLE_CHANNELS = [
    "LFootProgressAngles", "RFootProgressAngles",
    "LHipAngles",          "RHipAngles",
    "LKneeAngles",         "RKneeAngles",
    "LAnkleAngles",        "RAnkleAngles",
    "LForeFootAngles",     "RForeFootAngles",
    "LSpineAngles",        "RSpineAngles",
    "LShoulderAngles",     "RShoulderAngles",
    "LElbowAngles",        "RElbowAngles",
    "LWristAngles",        "RWristAngles",
    "LNeckAngles",         "RNeckAngles",
    "LPelvisAngles",       "RPelvisAngles",
    "LThoraxAngles",       "RThoraxAngles",
    "LHeadAngles",         "RHeadAngles",
]

SEX_MAP = {
    "AJ026": "F", "BD004": "F", "BL025": "M", "CF027": "M", "CL007": "F",
    "DA013": "M", "DC005": "M", "GF022": "F", "GM001": "M", "GQ016": "M",
    "HN021": "F", "HS018": "F", "JS009": "F", "LA014": "F", "LD002": "M",
    "LF003": "M", "LL011": "M", "LM012": "M", "MR015": "M", "OB010": "M",
    "PC008": "F", "PQ006": "M", "RC020": "F", "RC023": "F", "RV028": "M",
    "SA017": "F", "SM019": "F", "TB030": "M", "TK029": "F", "YX024": "M"
}


TARGET_FRAMES = 100
MIN_CYCLE_FRAMES = 30 
MAX_CYCLE_FRAMES = 250  


def extract_subject_session(filepath):
    path_str = str(filepath).replace("\\", "/")
    parts = path_str.split("/")
    
    subject_id = None
    for part in parts:
        if part in SEX_MAP:
            subject_id = part
            break
    
    session_num = None
    for part in parts:
        if part.startswith("Session"):
            try:
                session_num = int(part.replace("Session", ""))
            except:
                pass
            break
    
    return subject_id, session_num


def find_foot_strikes(c3d_file, side="Right"):

    c3d = ezc3d.c3d(str(c3d_file))
    
    events = c3d["parameters"]["EVENT"]
    num_events = int(events["USED"]["value"][0])
    
    if num_events == 0:
        return []
    
    labels = [str(x).strip() for x in events["LABELS"]["value"][:num_events]]
    contexts = [str(x).strip() for x in events["CONTEXTS"]["value"][:num_events]]
    times_sec = events["TIMES"]["value"][1][:num_events]
    
    fps = float(c3d["parameters"]["POINT"]["RATE"]["value"][0])
    first_frame = int(c3d["header"]["points"]["first_frame"])
    last_frame = int(c3d["header"]["points"]["last_frame"])
    num_frames = last_frame - first_frame + 1
    
    fs_frames = []
    side_lower = side.lower()
    
    for i in range(num_events):
        label = labels[i].lower()
        context = contexts[i].lower()
        
        is_strike = "strike" in label or "contact" in label
        is_correct_side = side_lower in context
        
        if is_strike and is_correct_side:
            time_sec = times_sec[i]
            
            frame_local = int(time_sec * fps)
            frame_global = int(time_sec * fps) - first_frame
            
            if 0 <= frame_local < num_frames:
                fs_frames.append(frame_local)
            elif 0 <= frame_global < num_frames:
                fs_frames.append(frame_global)
    
    return sorted(fs_frames)


def load_angle_data(c3d_file, channel_names):
    c3d = ezc3d.c3d(str(c3d_file))
    
    all_labels = [str(x) for x in c3d["parameters"]["POINT"]["LABELS"]["value"]]
    
    for ch in channel_names:
        if ch not in all_labels:
            raise ValueError(f"Kanał {ch} nie istnieje w pliku")
    
    channel_indices = [all_labels.index(ch) for ch in channel_names]
    points = c3d["data"]["points"]
    angle_data = points[:3, channel_indices, :]
    angle_data = np.transpose(angle_data, (2, 1, 0))
    
    return angle_data


def normalize_cycle(cycle_data, target_frames=100):
    num_frames, num_channels, num_axes = cycle_data.shape
    normalized = np.zeros((target_frames, num_channels, num_axes), dtype=np.float32)
    
    old_time = np.arange(num_frames)
    new_time = np.linspace(0, num_frames - 1, target_frames)
    
    for ch in range(num_channels):
        for ax in range(num_axes):
            values = cycle_data[:, ch, ax]
            interpolator = interp1d(old_time, values, kind='linear')
            normalized[:, ch, ax] = interpolator(new_time)
    
    return normalized


def extract_cycles_from_file(c3d_file, activity_type):

    subject_id, _ = extract_subject_session(c3d_file)
    
    
    # Znajdź Foot Strike eventy
    fs_frames = find_foot_strikes(c3d_file, side="Right")
    
    if len(fs_frames) < 2:
        return []
    
    # Wczytaj dane
    try:
        angle_data = load_angle_data(c3d_file, ANGLE_CHANNELS)
    except Exception as e:
        print(f"    x Błąd: {e}")
        return []
    
    total_frames = angle_data.shape[0]
    
    # Wytnij cykle
    cycles = []
    for i in range(len(fs_frames) - 1):
        start = fs_frames[i]
        end = fs_frames[i + 1]
        
        if start < 0 or end >= total_frames:
            continue
        
        cycle_length = end - start
        
        if cycle_length < MIN_CYCLE_FRAMES or cycle_length > MAX_CYCLE_FRAMES:
            continue
        
        cycle = angle_data[start:end, :, :]
        
        if np.isnan(cycle).any():
            continue
        
        cycle_normalized = normalize_cycle(cycle, TARGET_FRAMES)
        cycles.append(cycle_normalized)
    
    return cycles



def collect_data_for_activity(activity_type):

        pattern = "**/Overground_Walk/Walk_Comfortable/Post_Process/*.c3d"
    elif activity_type == "run":
        pattern = "**/Overground_Run/Run_Comfortable/Post_Process/*.c3d"
    else:
        raise ValueError(f"Nieznany activity_type: {activity_type}")
    
    all_files = sorted(ROOT.rglob(pattern))
    print(f"Znaleziono {len(all_files)} plików C3D")
    
    if len(all_files) == 0:
        print(f" BRAK PLIKÓW Sprawdź ścieżkę.")
        return None
    
    all_cycles = []
    all_subjects = []
    all_sessions = []
    all_sexes = []
    
    for filepath in all_files:
        subject_id, session_num = extract_subject_session(filepath)
        
        if subject_id is None or session_num is None:
            continue
        
        sex = SEX_MAP.get(subject_id)
        if sex is None:
            continue
        
        cycles = extract_cycles_from_file(filepath, activity_type)
        
        if len(cycles) == 0:
            continue
        
        for cycle in cycles:
            all_cycles.append(cycle)
            all_subjects.append(subject_id)
            all_sessions.append(session_num)
            all_sexes.append(sex)
        
        print(f"  git {subject_id}/S{session_num}: {len(cycles)} cykli")
    
    if len(all_cycles) == 0:
        print(f" Brak zebranych cykli")
        return None
    
    # Konwertuj do numpy
    X = np.array(all_cycles, dtype=np.float32)  # (N, 100, 26, 3)
    X_flat = X.reshape(X.shape[0], TARGET_FRAMES, -1)  # (N, 100, 78)
    
    print(f"\n-- Zebrano {len(X_flat)} cykli, shape: {X_flat.shape}")
    
    return {
        "X": X_flat,
        "X_3D": X,
        "subjects": np.array(all_subjects),
        "sessions": np.array(all_sessions),
        "sexes": np.array(all_sexes),
        "activity": activity_type
    }


def preprocess_walk_vs_run():

    walk_data = collect_data_for_activity("walk")
    
    run_data = collect_data_for_activity("run")
    
    if walk_data is None or run_data is None:
        print("\nNie udało się zebrać danych")
        return
    
    # WALK
    walk_train_mask = walk_data["sessions"] == 1
    walk_test_mask = walk_data["sessions"] == 2
    
    X_walk_train = walk_data["X"][walk_train_mask]
    X_walk_test = walk_data["X"][walk_test_mask]
    subjects_walk_train = walk_data["subjects"][walk_train_mask]
    subjects_walk_test = walk_data["subjects"][walk_test_mask]
    
    
    # RUN
    run_train_mask = run_data["sessions"] == 1
    run_test_mask = run_data["sessions"] == 2
    
    X_run_train = run_data["X"][run_train_mask]
    X_run_test = run_data["X"][run_test_mask]
    subjects_run_train = run_data["subjects"][run_train_mask]
    subjects_run_test = run_data["subjects"][run_test_mask]
    
    print(f"\n RUN:")
    print(f"  Train (S1): {X_run_train.shape} - {len(np.unique(subjects_run_train))} osób")
    print(f"  Test (S2):  {X_run_test.shape} - {len(np.unique(subjects_run_test))} osób")
    
    
    # 5. Zapis danych
    
    output_dir = ROOT / "WALK_VS_RUN_DATASET"
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Walk
    np.save(output_dir / "X_walk_train.npy", X_walk_train)
    np.save(output_dir / "X_walk_test.npy", X_walk_test)
    np.save(output_dir / "subjects_walk_train.npy", subjects_walk_train)
    np.save(output_dir / "subjects_walk_test.npy", subjects_walk_test)
    
    # Run
    np.save(output_dir / "X_run_train.npy", X_run_train)
    np.save(output_dir / "X_run_test.npy", X_run_test)
    np.save(output_dir / "subjects_run_train.npy", subjects_run_train)
    np.save(output_dir / "subjects_run_test.npy", subjects_run_test)
    
    # Metadane
    metadata = {
        "channels": ANGLE_CHANNELS,
        "num_channels": len(ANGLE_CHANNELS),
        "num_axes": 3,
        "num_features": len(ANGLE_CHANNELS) * 3,
        "target_frames": TARGET_FRAMES,
        "walk_train_shape": X_walk_train.shape,
        "walk_test_shape": X_walk_test.shape,
        "run_train_shape": X_run_train.shape,
        "run_test_shape": X_run_test.shape,
    }
    np.save(output_dir / "metadata.npy", metadata, allow_pickle=True)
    
    return {
        "walk_train": X_walk_train,
        "walk_test": X_walk_test,
        "run_train": X_run_train,
        "run_test": X_run_test,
        "output_dir": output_dir
    }


if __name__ == "__main__":
    data = preprocess_walk_vs_run()